In [3]:
import pandas as pd
import joblib
import glob
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# 1. LOAD AND MERGE ALL 5 CSV FILES
# Assuming your files are in the current directory or a folder named 'dataset'
file_pattern = r"C:\Users\JAN\Downloads\beef_data\csv\TS*.csv"  # Matches TS1.csv, TS2.csv, etc.
all_files = glob.glob(file_pattern)

if not all_files:
    print("No CSV files found! Please check the file path.")
else:
    print(f"Found {len(all_files)} files: {all_files}")

li = []
for filename in all_files:
    df = pd.read_csv(filename)
    df.columns = df.columns.str.strip() # Clean column names
    li.append(df)

# Combine all files into one large dataframe
data = pd.concat(li, axis=0, ignore_index=True)

# 2. DEFINE FEATURES AND TARGET
TARGET_COL = "class"

# We explicitly list the sensors available in the NEW dataset (Excluding MQ7)
# This also ensures Temperature and Humidity are in a fixed order regardless of file structure
FEATURES = [
    'MQ135', 'MQ136', 'MQ2', 'MQ3', 'MQ4', 
    'MQ5', 'MQ6', 'MQ8', 'MQ9', 
    'Temperature', 'Humidity'
]

# Drop rows with missing labels
data.dropna(subset=[TARGET_COL], inplace=True)

# Fill missing numeric values using the mean of the combined dataset
data[FEATURES] = data[FEATURES].fillna(data[FEATURES].mean())

# 3. ENCODE CLASS LABELS
le = LabelEncoder()
data[TARGET_COL] = le.fit_transform(data[TARGET_COL])

print("\nUpdated Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# Separate Features (X) and Target (y)
X = data[FEATURES]
y = data[TARGET_COL]

# 4. TRAIN-TEST SPLIT (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. TRAIN UPDATED RANDOM FOREST
# Increased n_estimators slightly to 200 to handle the larger, more diverse data
model = RandomForestClassifier(
    n_estimators=200, 
    random_state=42,
    n_jobs=-1 # Uses all CPU cores for faster training on larger data
)
model.fit(X_train, y_train)

# 6. EVALUATE
y_pred = model.predict(X_test)
print("\n--- Model Evaluation ---")
print(f"Accuracy with combined dataset: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

# 7. SAVE NEW MODEL AND ENCODER
joblib.dump(model, "updated_beef_rf_model.pkl")
joblib.dump(le, "updated_beef_label_encoder.pkl")

# Save the list of feature names to ensure the prediction script uses the same order
joblib.dump(FEATURES, "model_features.pkl")

print("\nNew model trained on all 5 files and saved successfully.")

Found 5 files: ['C:\\Users\\JAN\\Downloads\\beef_data\\csv\\TS1.csv', 'C:\\Users\\JAN\\Downloads\\beef_data\\csv\\TS2.csv', 'C:\\Users\\JAN\\Downloads\\beef_data\\csv\\TS3.csv', 'C:\\Users\\JAN\\Downloads\\beef_data\\csv\\TS4.csv', 'C:\\Users\\JAN\\Downloads\\beef_data\\csv\\TS5.csv']

Updated Class mapping: {'acceptable': np.int64(0), 'excellent': np.int64(1), 'good': np.int64(2), 'spoiled': np.int64(3)}

--- Model Evaluation ---
Accuracy with combined dataset: 0.9963
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       420
           1       0.99      1.00      1.00       336
           2       1.00      0.99      1.00       564
           3       1.00      1.00      1.00       840

    accuracy                           1.00      2160
   macro avg       1.00      1.00      1.00      2160
weighted avg       1.00      1.00      1.00      2160


New model trained on all 5 files and saved successfully.


In [4]:
import pandas as pd

importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importance)

MQ4            0.254436
MQ8            0.151617
MQ3            0.117352
Temperature    0.102434
MQ135          0.085861
MQ136          0.068799
MQ2            0.051240
Humidity       0.049630
MQ9            0.044993
MQ5            0.039762
MQ6            0.033877
dtype: float64


In [ ]:
import joblib
import pandas as pd
import numpy as np

# 1. LOAD THE TRAINED ASSETS
try:
    model = joblib.load("updated_beef_rf_model.pkl")
    le = joblib.load("updated_beef_label_encoder.pkl")
    features_list = joblib.load("model_features.pkl")
    print("--- Model Assets Loaded Successfully ---\n")
except FileNotFoundError:
    print("Error: Ensure the .pkl files are in the same folder as this script.")
    exit()

def run_mock_test():
    print("Manual Beef Quality Test")
    print("Enter the sensor values below (Use numbers only):")
    
    # 2. COLLECT MANUAL INPUTS
    user_inputs = {}
    for feature in features_list:
        try:
            val = float(input(f"Value for {feature}: "))
            user_inputs[feature] = val
        except ValueError:
            print("Invalid input. Please enter a numeric value.")
            return

    # 3. CONVERT TO DATAFRAME
    # This ensures the features are in the exact order the RF model learned
    test_df = pd.DataFrame([user_inputs], columns=features_list)

    # 4. PREDICT
    # Predict the numeric class
    prediction_id = model.predict(test_df)[0]
    
    # Get the probability/confidence for the prediction
    probabilities = model.predict_proba(test_df)[0]
    confidence = np.max(probabilities) * 100

    # 5. TRANSLATE RESULT
    # Use the LabelEncoder to turn the number back into a word
    result_text = le.inverse_transform([prediction_id])[0]

    # 6. DISPLAY RESULTS
    print("\n" + "="*30)
    print(f"ANALYSIS RESULT: {result_text.upper()}")
    print(f"CONFIDENCE: {confidence:.2f}%")
    print("="*30)

if __name__ == "__main__":
    while True:
        run_mock_test()
        cont = input("\nRun another test? (y/n): ").lower()
        if cont != 'y':
            break

--- Model Assets Loaded Successfully ---

Manual Beef Quality Test
Enter the sensor values below (Use numbers only):


Value for MQ135:  4.8


In [8]:
import pandas as pd
import numpy as np
import glob
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# --- STEP 1: Import and Combine all 5 CSVs ---
all_files = glob.glob("TS*.csv") # This finds all your CSV files
data_list = []

for filename in all_files:
    # Assuming your CSVs have columns: mq135, mq136... mq137, temp, hum, label
    temp_df = pd.read_csv(filename)
    data_list.append(temp_df)

# Merge everything into one giant table
df = pd.concat(data_list, ignore_index=True)

# Remove the columns that aren't physical sensor data
cols_to_ignore=['minute', 'TVC']
df = df.drop(columns=cols_to_ignore, errors='ignore')

#Clean headers (removes hidden spaces)
df.columns = df.columns.str.strip()

# --- STEP 2: Windowing (The Math Magic) ---
window_size = 50

# We create the window_id so every 50 rows get the same ID
df['window_id'] = np.arange(len(df)) // window_size

# --- STEP 3: Feature Engineering ---
# We group by the window and the label. 
# We calculate Mean, Std, and Max for every sensor column.
features = df.groupby(['window_id', 'class']).agg(['mean', 'std', 'max']).reset_index()

# Flatten the column names (e.g., mq135_mean, mq135_std, mq135_max)
features.columns = ['_'.join(col).strip('_') for col in features.columns.values]

# --- STEP 4: Cleaning & Splitting ---
# Drop the ID and separate the Target (Label) from Features (X)
X = features.drop(['window_id', 'class'], axis=1)
y = features['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- STEP 5: Training ---
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# --- STEP 6: Save Everything for Flutter ---
joblib.dump(model, 'rf_model_33.pkl')
joblib.dump(list(X.columns), 'model_feature_33.pkl') # Very important for Dart order!

print(f"Model trained! Input features: {len(X.columns)}")
print(f"Accuracy on test set: {model.score(X_test, y_test):.4f}")

Model trained! Input features: 33
Accuracy on test set: 0.8723


In [9]:
import pandas as pd

importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importance)

MQ4_max             0.112628
MQ4_mean            0.107975
MQ8_mean            0.078260
MQ4_std             0.072241
MQ8_max             0.062771
MQ3_mean            0.055194
MQ3_max             0.051018
MQ135_mean          0.037549
Temperature_mean    0.034559
Temperature_max     0.034558
MQ8_std             0.032320
MQ135_max           0.030631
MQ6_std             0.025903
MQ2_mean            0.024148
MQ9_mean            0.018296
MQ136_mean          0.017693
MQ6_mean            0.017212
MQ5_mean            0.016480
MQ5_max             0.015661
MQ2_max             0.014821
MQ2_std             0.014439
MQ135_std           0.014356
MQ136_max           0.014240
MQ3_std             0.014109
Humidity_mean       0.012715
Humidity_max        0.011957
MQ9_max             0.010798
MQ6_max             0.010788
MQ5_std             0.009441
Temperature_std     0.008658
MQ136_std           0.007249
MQ9_std             0.007144
Humidity_std        0.004191
dtype: float64


In [10]:
import joblib

# Load the feature list you saved during training
features = joblib.load('model_feature_33.pkl')

print("--- DART ARRAY ORDER (Copy this) ---")
for i, name in enumerate(features):
    print(f"Index {i:2}: {name}")

print("\n--- READY-TO-USE DART LIST ---")
print("List<double> featureVector = [")
for name in features:
    # This cleans the name for Dart (e.g., mq135_mean -> mq135Mean)
    clean_name = name.replace('_m', 'M').replace('_s', 'S').replace('temp', 'Temp').replace('hum', 'Hum')
    print(f"  {clean_name}, // {name}")
print("];")

--- DART ARRAY ORDER (Copy this) ---
Index  0: MQ135_mean
Index  1: MQ135_std
Index  2: MQ135_max
Index  3: MQ136_mean
Index  4: MQ136_std
Index  5: MQ136_max
Index  6: MQ2_mean
Index  7: MQ2_std
Index  8: MQ2_max
Index  9: MQ3_mean
Index 10: MQ3_std
Index 11: MQ3_max
Index 12: MQ4_mean
Index 13: MQ4_std
Index 14: MQ4_max
Index 15: MQ5_mean
Index 16: MQ5_std
Index 17: MQ5_max
Index 18: MQ6_mean
Index 19: MQ6_std
Index 20: MQ6_max
Index 21: MQ8_mean
Index 22: MQ8_std
Index 23: MQ8_max
Index 24: MQ9_mean
Index 25: MQ9_std
Index 26: MQ9_max
Index 27: Humidity_mean
Index 28: Humidity_std
Index 29: Humidity_max
Index 30: Temperature_mean
Index 31: Temperature_std
Index 32: Temperature_max

--- READY-TO-USE DART LIST ---
List<double> featureVector = [
  MQ135Mean, // MQ135_mean
  MQ135Std, // MQ135_std
  MQ135Max, // MQ135_max
  MQ136Mean, // MQ136_mean
  MQ136Std, // MQ136_std
  MQ136Max, // MQ136_max
  MQ2Mean, // MQ2_mean
  MQ2Std, // MQ2_std
  MQ2Max, // MQ2_max
  MQ3Mean, // MQ3_mean
  